# A02: Asincronismo con Asyncio

**Nivel:** Avanzado  |  **Tema:** Programación asíncrona con `asyncio` en Python

Python 3.12+ · Librerías: `asyncio` (stdlib), opcional `httpx`/`aiohttp`

---

> La programación asíncrona te permite **esperar** sin **bloquear**. Un programa async puede
> avanzar en varias tareas mientras una de ellas está "en pausa" esperando I/O (red, disco, usuario).

## Objetivos

Al terminar este notebook serás capaz de:

1. **Explicar** qué es el event loop y diferenciar corutinas, tareas y awaitables.
2. **Escribir** funciones `async def` y ejecutarlas con `await`, `asyncio.run()` y `asyncio.create_task()`.
3. **Combinar** corutinas concurrentemente con `asyncio.gather()` y `asyncio.wait()`.
4. **Controlar** concurrencia con `asyncio.Semaphore`, sincronizar con `Lock` y `Queue`, y aplicar timeouts.
5. **Construir** protocolos asíncronos (iteradores, generadores, context managers) y aplicarlos a un download manager real.

---

## Analogía: el mesero de restaurante

Imagina un **mesero** atendiendo varias mesas. Cuando la mesa 1 hace un pedido, el cocinero
tarda en prepararlo. El mesero **no se queda parado** mirando la cocina: aprovecha ese tiempo
para atender la mesa 2, la 3, etc.

- `asyncio` = el mesero (una sola persona, no varias).
- `await` = "mientras esto se prepara, puedo hacer otra cosa".
- El **cocinero** = operación de I/O (red, disco) que tarda.
- Cuando la comida de la mesa 1 está lista, el mesero **retoma** y se la lleva.

```
Mesa 1 ----- await(pedido) -------  retoma  ------- entrega
     \          \                      /
Mesa 2 --- pedido ---- espera ---- retoma -------- entrega
     \        \              /
Mesa 3 - pedido -- espera -- retoma -------- entrega

Tiempo ---->  (el mesero nunca deja de trabajar; es UNO SOLO)
```

Un solo mesero hace más trabajo útil en el mismo tiempo porque **esperar es gratis**
(el CPU no se queda parado). Así es `asyncio`.

## 1. Conceptos fundamentales

### El event loop

El **event loop** es el núcleo de `asyncio`: un bucle que gestiona y reparte eventos
("el pedido de la mesa 1 está listo", "la red respondió", "pasaron 2 segundos").

```
                    ┌─────────────────────┐
   Tarea A ───────▶ │                     │
   Tarea B ───────▶ │    EVENT LOOP       │  ──▶ ejecuta cuando está listo
   Timer   ───────▶ │   (mesero / cpu)    │
   Socket  ───────▶ │                     │
                    └─────────────────────┘
```

### Vocabulario esencial

| Término | Definición | Ejemplo |
|---|---|---|
| **Corutina** | Función declarada con `async def`; NO se ejecuta al llamarla | `async def foo(): ...` |
| **Awaitable** | Objeto que puede esperarse con `await` (corutina, Task, Future) | `await foo()` |
| **Task** | Corutina *programada* en el event loop para ejecución concurrente | `t = asyncio.create_task(foo())` |
| **Event loop** | Bucle que orquesta cuándo corre cada tarea | `asyncio.run(main())` |

### Async ≠ Paralelo

- **Paralelismo** (multithreading/multiprocessing): varias cosas *a la vez*, en varios hilos/núcleos.
- **Asincronía** (`asyncio`): **una** cosa a la vez, pero cambiando de tarea cuando hay espera (I/O).

`asyncio` usa **concurrencia** (muchas tareas adminando *intercaladas*), NO paralelismo real
(solo aprovecha un hilo). Es ideal para problemas **ligados a I/O**, no para cómputo puro.

### Cooperativo vs Preemptivo

- **Preemptivo** (threads): el sistema operativo interrumpe un hilo cuando se le antoja (problemas de *race conditions*).
- **Cooperativo** (`asyncio`): cada corutina **cede el control** voluntariamente con `await`. Solo cambias
  de tarea en un `await`, lo que elimina muchas condiciones de carrera, pero exige que **tú** cedas
  (nunca llamar código bloqueante).

## 2. async/await básico

Una función `async def` devuelve una **corutina** (objeto), NO su resultado. Al llamarla
nada se ejecuta todavía. Hay que **programarla** con `await`, `asyncio.run()` o `create_task()`.

Veamos la diferencia entre *llamar* y *ejecutar*:

In [ ]:
async def saludar(nombre: str) -> str:
    return f"Hola, {nombre}"

# Llamar a la corutina NO ejecuta el código:
corutina = saludar("Ana")
print(type(corutina))
print(repr(corutina))

# Para ejecutarla, la 'awaitamos':
import asyncio
print(asyncio.run(saludar("Ana")))

Observa que `type(corutina)` es `coroutine` (la función no se ha ejecutado).
El código dentro de `async def` solo corre cuando se espera con `await` (directamente)
o se programa en el loop (`asyncio.run`, `create_task`).

> ⚠️ **Regla de oro:** una corutina sin consumir genera un *warning* `RuntimeWarning:
> coroutine ... was never awaited`. Siempre ejecútala.

In [ ]:
import asyncio

async def desayunar():
    print("1. Pongo la cafetera")
    await asyncio.sleep(1)          # ← aquí 'cedo' el control durante 1s
    print("2. Café listo")
    return "café ☕"

async def main():
    resultado = await desayunar()   # la corutina corre y esperamos su resultado
    print("Resultado:", resultado)

await main()

### La clave del interleaving

Cuando una corutina ejecuta `await`, el event loop **guarda su estado** y puede pasar
a otra tarea. Eso permite que varias corutinas "avancen" simultáneamente sin bloquear.

```
corutina A:  print(...) ─ await ───────────────── print(...)
corutina B:               print(...) ─ await ─ print(...)

tiempo ──────────────────────────────────────────────────▶
  (ambas avanzan intercaladas, mientras A/B esperan su sleep)
```

## 3. Ejecutar corutinas: `asyncio.run`, `get_event_loop`, `create_task`

Tres formas de ejecutar corutinas:

| Forma | Cuándo usarla |
|---|---|
| `asyncio.run(coro)` | Punto de entrada principal; crea/cierra un loop de forma segura |
| `await coro` | Dentro de otra corutina, para esperar su resultado |
| `asyncio.create_task(coro)` | Programar una corutina para que corra **concurrentemente** (sin esperarla al instante) |

`asyncio.run()` crea un nuevo event loop, lo ejecuta y lo cierra. Es la forma recomendada
desde Python 3.7. `loop = asyncio.get_event_loop()` se usa en código legacy; en 3.12
si no hay loop corriendo lanza `RuntimeError`.

In [ ]:
import asyncio

async def tarea(nombre: str, delay: float) -> str:
    await asyncio.sleep(delay)
    return f"{nombre} terminó tras {delay}s"

async def main():
    # 1) await directo: esperamos UNA corutina
    r1 = await tarea("A", 0.5)
    print(r1)

    # 2) create_task: programamos varias para que corran 'a la vez'
    t1 = asyncio.create_task(tarea("B", 0.3))
    t2 = asyncio.create_task(tarea("C", 0.2))
    r2 = await t1
    r3 = await t2
    print(r2)
    print(r3)

asyncio.run(main())
print("--- fin ---")

Nota importante: con `create_task`, `B` y `C` se programan y **después** las esperamos.
Como `B` duerme 0.3s y `C` 0.2s, `C` termina antes aunque la creada primero sea `B`.
Ambas corren **intercaladas** en el mismo loop, en paralelo *lógico* (no multicore).

### Ejecución concurrente real

Para ver el beneficio del tiempo total, comparamos secuencial vs concurrente:


In [ ]:
import asyncio
import time

async def descargar(nombre: str, seg: float) -> str:
    await asyncio.sleep(seg)
    return nombre

async def secuencial():
    for d in [1, 1, 1]:
        await descargar("archivo", d)   # se espera cada uno antes del siguiente

async def concurrente():
    tareas = [descargar("archivo", 1) for _ in range(3)]
    await asyncio.gather(*tareas)        # los 3 'descargando' a la vez

def cronometrar(coro_factory):
    inicio = time.perf_counter()
    asyncio.run(coro_factory())
    return time.perf_counter() - inicio

t_sec = cronometrar(secuencial)
t_conc = cronometrar(concurrente)

print(f"Secuencial  : {t_sec:.2f}s  (acumula 3×1s)")
print(f"Concurrente : {t_conc:.2f}s  (corren a la vez, ≈1s)")
print(f"Ganancia    : {t_sec / t_conc:.1f}x más rápido")

## 4. `asyncio.gather` y `asyncio.wait`

### `asyncio.gather`

Ejecuta varias corutinas **concurrentemente** y devuelve sus resultados **en el orden de
los argumentos** (no en el orden de finalización).



In [ ]:
import asyncio

async def operacion(nombre: str, delay: float) -> str:
    await asyncio.sleep(delay)
    return nombre.upper()

async def main():
    resultados = await asyncio.gather(
        operacion("alfa", 0.4),   # termina de última pero va primero en resultado
        operacion("beta", 0.1),
        operacion("gamma", 0.2),
    )
    # Los resultados están en el ORDEN de llamada, no de finalización:
    print("Resultados en orden de declaración:", resultados)

asyncio.run(main())

In [ ]:
import asyncio

# gather CON manejo de excepciones
async def puede_fallar(nombre: str, ok: bool):
    await asyncio.sleep(0.1)
    if not ok:
        raise ValueError(f"{nombre} falló")
    return nombre

async def main():
    try:
        await asyncio.gather(
            puede_fallar("x", True),
            puede_fallar("y", False),
            puede_fallar("z", True),
        )
    except ValueError as e:
        print("Capturada:", e)

    # Con return_exceptions=True NO lanza; devuelve la excepción como resultado
    res = await asyncio.gather(
        puede_fallar("x", True),
        puede_fallar("y", False),
        return_exceptions=True,
    )
    print("return_exceptions=True →", res)

asyncio.run(main())

### `asyncio.wait`

A diferencia de `gather`, `wait` trabaja con **Tasks** y permite esperar según cuándo
terminen, gracias al parámetro `return_when`: `ALL_COMPLETED`, `FIRST_COMPLETED`, `FIRST_EXCEPTION`.

In [ ]:
import asyncio

async def trabajo(nombre: str, seg: float):
    await asyncio.sleep(seg)
    return nombre

async def main():
    tareas = [
        asyncio.create_task(trabajo("A", 0.5)),
        asyncio.create_task(trabajo("B", 1.0)),
    ]

    # Esperamos hasta que la PRIMERA termine
    done, pending = await asyncio.wait(tareas, return_when=asyncio.FIRST_COMPLETED)
    print("Terminadas primero:", [t.result() for t in done])
    print("Pendientes (siguen corriendo):", len(pending))

    # Esperamos el resto
    done2, _ = await asyncio.wait(pending)
    print("Y las demás:", sorted(t.result() for t in done2))

asyncio.run(main())

In [ ]:
import asyncio

async def main():
    t = asyncio.create_task(trabajo_rapido())
    done, _ = await asyncio.wait({t}, timeout=2.0)
    print("¿Terminó a tiempo?", bool(done))

async def trabajo_rapido():
    await asyncio.sleep(1.0)
    return "ok"

asyncio.run(main())

## Diagrama: flujo de `asyncio.gather`

```
asyncio.gather(a(), b(), c())

        a() ──▶ [inicio] ── await ──▶ [fin] ──┐
        b() ──▶ [inicio] ── await ────────────┼──▶ recogidos en orden a,b,c
        c() ──▶ [inicio] ── await ────────────┘        (no por fin de ejecución)

Event loop intercala los awaits para que los 3 avancen a la vez.
```


## 5. Estructuras asíncronas

`asyncio` aporta primitivas de sincronización y control parecidas a `threading`, pero
adaptadas al mundo cooperativo.

### `asyncio.Semaphore` — limitar concurrencia (rate limiting)

Si lanzas 1000 descargas `gather`, todas intentan correr a la vez (y el servidor o tu
máquina lo pagan). Un **semáforo** limita cuántas corutinas pueden pasar al mismo tiempo.



In [ ]:
import asyncio

async def descargar(id_, semaforo: asyncio.Semaphore):
    async with semaforo:                       # máx. 2 a la vez
        print(f"▶ descargando {id_}")
        await asyncio.sleep(0.3)
        print(f"✔ descargado {id_}")

async def main():
    limite = asyncio.Semaphore(2)              # 2 descargas concurrentes máx.
    tareas = [descargar(i, limite) for i in range(6)]
    await asyncio.gather(*tareas)

asyncio.run(main())

Observa cómo **nunca hay más de 2** descargas simultáneas: los prints `▶` aparecen de dos
en dos. Eso es *rate limiting*. Sin semáforo, las 6 saldrían a la vez.

### `asyncio.Lock` — exclusión mutua

Garantiza que solo **una** corutina entre a la sección crítica a la vez. Útil para
recursos compartidos (archivos, contadores).


In [ ]:
import asyncio

async def incrementar(lock, contador: dict, id_):
    async with lock:                            # sección crítica protegida
        actual = contador["valor"]
        await asyncio.sleep(0.05)               # simula operación costosa
        contador["valor"] = actual + 1

async def main():
    lock = asyncio.Lock()
    contador = {"valor": 0}
    await asyncio.gather(*[incrementar(lock, contador, i) for i in range(20)])
    print("Contador final (debe ser 20):", contador["valor"])

asyncio.run(main())

Sin el `Lock`, como entre leer y escribir hay un `await`, otra corutina podría leer el
mismo valor y perderíamos incrementos. El `Lock` garantiza atomicidad lógica.

### `asyncio.Queue` — patrón productor–consumidor

Una cola FIFO segura para pasar trabajo de productores a consumidores, evitando
consumidores ociosos o trabajo duplicado.


In [ ]:
import asyncio

async def productor(queue: asyncio.Queue, n: int):
    for i in range(n):
        print(f"🏭 produciendo item {i}")
        await queue.put(i)
        await asyncio.sleep(0.05)
    await queue.put(None)             # señal de fin

async def consumidor(queue: asyncio.Queue, nombre: str):
    while True:
        item = await queue.get()
        if item is None:
            await queue.task_done()
            break
        print(f"🛠 {nombre} procesa item {item}")
        await asyncio.sleep(0.2)
        await queue.task_done()

async def main():
    q = asyncio.Queue()
    await asyncio.gather(
        productor(q, 5),
        consumidor(q, "c1"),
        consumidor(q, "c2"),
    )
    print("Cola vacía y procesada ✔")

asyncio.run(main())

### Timeouts: `asyncio.wait_for` y `asyncio.timeout` (3.11+)

Para no esperar por siempre una operación que no responde, ponemos un límite de tiempo.

In [ ]:
import asyncio

async def lento():
    await asyncio.sleep(10)
    return "nunca llegará a tiempo"

async def main():
    # estilo clásico: asyncio.wait_for
    try:
        await asyncio.wait_for(lento(), timeout=1.0)
    except asyncio.TimeoutError:
        print("¡wait_for: excedió 1 segundo!")

    # estilo moderno (3.11+): asyncio.timeout como context manager
    try:
        async with asyncio.timeout(1.0):
            await lento()
    except TimeoutError:
        print("¡asyncio.timeout: límite excedido!")

asyncio.run(main())

## Tabla de referencia: estructuras asíncronas

| Estructura | Uso | Método clave | Analogía |
|---|---|---|---|
| `Semaphore(n)` | Limitar # de corutinas concurrentes | `async with sem:` | aforo del local |
| `Lock()` | Exclusión mutua de sección crítica | `async with lock:` | cerrojo del baño |
| `Queue()` | Cola FIFO segura productor/consumidor | `await q.put()/get()` | fila del supermercado |
| `Event()` | Señalar que algo ocurrió | `await evt.wait()` | semáforo de tráfico |
| `timeout()` | Límite de tiempo | `async with asyncio.timeout(s)` | cronómetro |


## 6. Protocolos asíncronos

Python soporta versiones **asíncronas** de sus protocolos de iteración, generación y
contexto, identificables por el prefijo `async` y los métodos `__a...__`.

### 6.1 Iteradores asíncronos (`async for`)

Un objeto iterable que produce los siguientes elementos tras **esperas** de I/O.


In [ ]:
import asyncio

class NumerosAsync:
    """Iterador asíncrono: genera números tras una espera de I/O."""
    def __init__(self, n):
        self.n = n
        self.i = 0

    def __aiter__(self):
        return self

    async def __anext__(self):
        if self.i >= self.n:
            raise StopAsyncIteration
        await asyncio.sleep(0.1)      # simula leer datos que llegan poco a poco
        self.i += 1
        return self.i

async def main():
    async for num in NumerosAsync(3):
        print("Recibido:", num)

asyncio.run(main())

### 6.2 Generadores asíncronos (`async def` + `yield`)

Como un generador normal, pero puede `await` entre `yield`s. Mucho más simple que
implementar `__aiter__`/`__anext__` a mano.

In [ ]:
import asyncio

async def flujo_datos(topo: int):
    for dato in range(1, topo + 1):
        await asyncio.sleep(0.1)      # simula llegada por red
        yield dato                    # entrega el dato; se puede pausar aquí

async def main():
    async for d in flujo_datos(3):
        print("Dato del stream:", d)

asyncio.run(main())

### 6.3 Context managers asíncronos (`async with`)

Cuando abrir/cerrar un recurso implica esperas (conexiones, archivos en red, sesiones HTTP),
usamos `__aenter__` / `__aexit__` y `async with`.

In [ ]:
import asyncio

class Conexion:
    async def __aenter__(self):
        print("▶ conectando...")
        await asyncio.sleep(0.2)
        print("✔ conectado")
        return self

    async def __aexit__(self, exc_type, exc, tb):
        print("▶ cerrando conexión...")
        await asyncio.sleep(0.1)
        print("✔ conexión cerrada")

    async def consulta(self):
        await asyncio.sleep(0.1)
        return "datos"

async def main():
    async with Conexion() as conn:
        print("Resultado consulta:", await conn.consulta())

asyncio.run(main())

### 6.4 Comprensiones asíncronas

Podemos combinar `async for` dentro de listas, dicts y sets comprensivos, siempre dentro
de una corutina.


In [ ]:
import asyncio

async def obtener(n: int) -> int:
    await asyncio.sleep(0.05)
    return n * 10

async def main():
    # comprensión asíncrona: await dentro de lista
    resultados = [await obtener(i) for i in range(3)]
    print("Comprensión con await:", resultados)

    # comprensión combinada con async generator
    async def gen():
        for i in range(3):
            yield i
    duplicados = [x * 2 async for x in gen()]
    print("Comprensión async-for :", duplicados)

asyncio.run(main())

## 7. Comunicación de red real (httpx / aiohttp)

El caso de uso estrella de `asyncio` es **I/O de red**: descargar cientos de URLs sin esperar
una por una. Si tienes `httpx` o `aiohttp` instalados, la estructura es la siguiente.

Primero, verifiquemos qué librería está disponible en tu entorno:

In [ ]:
def libreria_http():
    """Devuelve el nombre de la librería async HTTP disponible (o None)."""
    try:
        import httpx
        return "httpx"
    except ImportError:
        try:
            import aiohttp
            return "aiohttp"
        except ImportError:
            return None

print("Librería disponible:", libreria_http())

### Patrón real con `httpx`

`httpx` es la librería HTTP moderna, con cliente **asíncrono** (`httpx.AsyncClient`).
El patrón es: abrir un cliente con `async with`, y disparar N peticiones con `gather`.

```python
import asyncio
import httpx

URLS = ["https://api.github.com", "https://httpbin.org/json", "https://example.com"]

async def fetch(client: httpx.AsyncClient, url: str) -> tuple[str, int]:
    resp = await client.get(url)          # ← I/O de red real, libera el loop
    return url, resp.status_code

async def main():
    async with httpx.AsyncClient(timeout=10.0) as client:
        resultados = await asyncio.gather(
            *(fetch(client, u) for u in URLS)
        )
    for url, code in resultados:
        print(f"{url} → {code}")

asyncio.run(main())
```

Si no tienes `httpx`: `pip install httpx`.

### Alternativa con `aiohttp`

```python
import aiohttp
import asyncio

async def fetch(session, url):
    async with session.get(url) as resp:
        return url, resp.status

async def main():
    async with aiohttp.ClientSession() as session:
        resultados = await asyncio.gather(*(fetch(session, u) for u in URLS))

asyncio.run(main())
```

Instalación: `pip install aiohttp` (incluye aiohttp solo, httpx incluye HTTP/2 y soporte sync).



### Simulación offline (si httpx/aiohttp NO están instalados)

Si tu entorno no tiene librería HTTP, replicamos el **mismo patrón de concurrencia** usando
`asyncio.sleep` como sustituto del tiempo de red. La estructura (cliente → gather → semáforo)
es idéntica a la real.

In [ ]:
import asyncio

URLS = ["https://site-a.example", "https://site-b.example", "https://site-c.example"]

async def fetch_simulada(url: str, retardo: float) -> tuple[str, int]:
    """Simula una petición HTTP: espera 'retardo' segundos y devuelve un status."""
    await asyncio.sleep(retardo)          # ← aquí iría el I/O real de red
    return url, 200

async def main():
    retardos = [0.3, 0.1, 0.2]            # cada URL tarda distinto
    resultados = await asyncio.gather(
        *(fetch_simulada(u, r) for u, r in zip(URLS, retardos))
    )
    for url, status in resultados:
        print(f"{url:35s} → HTTP {status}")

asyncio.run(main())
print("Todas las 'peticiones' se realizaron concurrentemente ✔")

In [ ]:
import asyncio
import time

# Mismo patrón PERO limitando concurrencia con un semáforo (rate limiting),
# como haría un cliente HTTP respetuoso.

async def fetch_con_limite(url: str, sem: asyncio.Semaphore):
    async with sem:                        # máx. 2 conexiones simultáneas
        await asyncio.sleep(0.2)           # simula tiempo de red
    return url, 200

async def main():
    sem = asyncio.Semaphore(2)
    inicio = time.perf_counter()
    res = await asyncio.gather(*(fetch_con_limite(u, sem) for u in range(6)))
    print(f"6 peticiones con límite de 2 → {time.perf_counter() - inicio:.2f}s")
    print("Errores HTTP:", [r for r in res if r[1] != 200])

asyncio.run(main())

## 8. Anti-patrones y errores comunes

Estos errores rompen o degradan silenciosamente el rendimiento async. Aprende a evitarlos.

### ❌ 1. Llamar `time.sleep()` dentro de una corutina

BLOQUEA TODO el event loop. Mientras duerme, ninguna otra tarea puede correr, destruyendo
todo el beneficio de la concurrencia. Siempre usa `asyncio.sleep()`.


In [ ]:
import asyncio
import time

# MAL: time.sleep bloquea el loop
async def malo():
    time.sleep(1)      # BLOQUEA: nadie más corre durante 1s

# BIEN: asyncio.sleep cede el control
async def bueno():
    await asyncio.sleep(1)   # libera el loop

print("Regla: en asyncio SIEMPRE await asyncio.sleep(), nunca time.sleep()")

### ❌ 2. Olvidar el `await`

Llamar a una corutina sin `await` crea un objeto coroutine que **nunca corre** y dispara
el warning `coroutine was never awaited`.

In [ ]:
import asyncio
import warnings

async def tarea():
    return 42

# FORZADO: AQUÍ OLVIDAMOS await (solo para mostrar el warning)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # capturamos para que no ensucie la salida
    objeto = tarea()
    print("Se creó el objeto coroutine, pero nunca se await-ó:", type(objeto).__name__)

print("Y la tarea NUNCA llegó a ejecutarse. → Siempre await, o crea la Task.")

### ❌ 3. Esperar secuencialmente dentro de `gather`

`asyncio.gather(*tareas)` espera a que **todas** terminen, pero debemos **construir** las
corutinas (no await-arlas) antes de pasarlas. Si dentro del generador haces `await fetch(...)`,
las ejecutas una a una y pierdes concurrencia.

In [ ]:
import asyncio
import time

async def trabajo(i):
    await asyncio.sleep(0.2)
    return i

# MAL: se crea con await → secuencial
async def mal():
    return [await trabajo(i) for i in range(4)]   # 4 × 0.2 = 0.8s en serie

# BIEN: se pasan las corutinas sín await → concurrente
async def bien():
    return await asyncio.gather(*(trabajo(i) for i in range(4)))  # ≈ 0.2s

def medir(coro):
    t = time.perf_counter(); asyncio.run(coro()); return time.perf_counter() - t

print(f"MAL (secuencial)  : {medir(mal):.2f}s")
print(f"BIEN (concurrente): {medir(bien):.2f}s")

### ❌ 4. No capturar excepciones en tasks desatendidas

Si creas una tarea con `create_task` y nunca la esperas ni lees `.result()`, una excepción
interna puede quedar silenciosa (o volcarse como *task exception was never retrieved*).
Captúrala siempre.

In [ ]:
import asyncio

async def frágil():
    await asyncio.sleep(0.1)
    raise ValueError("algo salió mal")

async def main():
    tarea = asyncio.create_task(frágil())
    try:
        await tarea            # la esperamos → la excepción se propaga y capturamos
    except ValueError as e:
        print("Capturada la excepción de la task:", e)

asyncio.run(main())

### ❌ 5. Reusar una corutina cerrada

Un objeto coroutine solo puede ejecutarse **una vez**. Tras el primer `await`, queda
cerrado (estado `CORO_CLOSED`) y no puede volverse a ejecutar. Para re-ejecutar la misma
lógica, llama de nuevo a la función `async def`.

In [ ]:
import asyncio

async def cuenta():
    return 7

async def main():
    coro = cuenta()
    print("Primer await:", await coro)
    print("Estado tras ejecutarla:", coro.cr_await, "→ está cerrada")

    # Para volver a hacerlo, llamamos OTRA VEZ a la función:
    print("Segunda llamada (nueva corutina):", await cuenta())

asyncio.run(main())

## Tabla de referencia: funciones `asyncio`

| Función | Qué hace | Uso típico |
|---|---|---|
| `asyncio.run(coro)` | Crea, ejecuta y cierra el loop | Entrada principal |
| `asyncio.create_task(coro)` | Programa una corutina para ejecución concurrente | Lanzar tareas en paralelo |
| `await coro` | Espera el resultado de una corutina | Dentro de otra corutina |
| `asyncio.gather(*coros)` | Ejecuta corutinas a la vez; resultados en orden | Tareas independientes |
| `asyncio.wait(tasks, return_when=...)` | Espera tasks según condición de finalización | `FIRST_COMPLETED`, etc. |
| `asyncio.sleep(s)` | Pausa sin bloquear; cede el loop | Simular I/O, delays |
| `asyncio.Semaphore(n)` | Limita concurrencia | Rate limiting |
| `asyncio.Lock()` | Exclusión mutua | Sección crítica |
| `asyncio.Queue()` | Cola FIFO productor/consumidor | Pipeline de trabajo |
| `asyncio.wait_for(coro, t)` | Lanza con timeout (clásico) | Evitar esperas infinitas |
| `asyncio.timeout(t)` | Timeout moderno (3.11+) | `async with asyncio.timeout(...)` |


## 9. Async en combos: `asyncio.run` / waiter

A menudo necesitas **combinar** múltiples tareas asíncronas y esperar su estado en varios
momentos: lanzar todo, procesar lo que vaya llegando y esperar el resto. Aquí combinamos
`create_task`, `asyncio.wait` con `FIRST_COMPLETED` y reintento de espera (patrón "waiter").



In [ ]:
import asyncio

async def trabajo_etapa(nombre: str, seg: float):
    await asyncio.sleep(seg)
    return f"{nombre} ({seg}s)"

async def waiter(tasks: set):
    """Va esperando y procesando a medida que terminan."""
    hechos = []
    while tasks:
        done, pending = await asyncio.wait(tasks, return_when=asyncio.FIRST_COMPLETED)
        hechos.extend(t.result() for t in done)
        tasks = pending        # AQUÍ esperamos el resto (patrón waiter) y volvemos al loop
    return hechos

async def main():
    tareas = {
        asyncio.create_task(trabajo_etapa("A", 0.4)),
        asyncio.create_task(trabajo_etapa("B", 0.2)),
        asyncio.create_task(trabajo_etapa("C", 0.7)),
    }
    orden_de_fin = await waiter(tareas)
    print("Orden en que terminaron:")
    for r in orden_de_fin:
        print("  ·", r)

asyncio.run(main())

Aquí el `waiter` itera con `wait(..., FIRST_COMPLETED)`, recoge lo que ya terminó y
sigue con lo pendiente. Nota la diferencia con `gather`, que solo devuelve resultados al
final de todas (y en orden de declaración). El patrón waiter te da **orden de finalización**.

## Ejercicio 1 (Guiado): Timeout defensivo

**Consigna:** Escribe una corutina `obtener_datos()` que tarde 2s, y un `main()` que la
llame con un timeout de 1s. Captura la excepción y muestra un mensaje.

**Pistas:** usa `asyncio.wait_for` y captura `asyncio.TimeoutError`.



In [ ]:
import asyncio

async def obtener_datos():
    await asyncio.sleep(2)
    return "datos completos"

async def main_timeout():
    try:
        datos = await asyncio.wait_for(obtener_datos(), timeout=1.0)
    except asyncio.TimeoutError:
        datos = "(timeout: se usó valor por defecto)"
    print("Datos:", datos)

asyncio.run(main_timeout())

## Ejercicio 2 (Guiado): Semáforo controlado

**Consigna:** Crea 8 corutinas `trabajo(i)` que duerman 0.2s, pero ejecútalas con un
`asyncio.Semaphore(3)` para que **nunca** haya más de 3 simultáneas. Usa un contador
compartido para verificar el máximo de concurrencia observado.



In [ ]:
import asyncio

async def trabajo_con_limite(i, sem, registro):
    async with sem:
        registro["activas"] += 1
        registro["max"] = max(registro["max"], registro["activas"])
        await asyncio.sleep(0.2)
        registro["activas"] -= 1
    return i

async def main():
    sem = asyncio.Semaphore(3)
    registro = {"activas": 0, "max": 0}
    resultados = await asyncio.gather(
        *(trabajo_con_limite(i, sem, registro) for i in range(8))
    )
    print("Resultados:", resultados)
    print("Concurrencia máxima observada:", registro["max"], "(debe ser ≤ 3)")

asyncio.run(main())

## Ejercicio 3 (Guiado): Productor–consumidor

**Consigna:** Usando `asyncio.Queue`, un productor genera 6 números y dos consumidores los
procesan. Usa la señal `None` al final para que los consumidores terminen limpiamente.



In [ ]:
import asyncio

async def productor(q):
    for n in range(6):
        await q.put(n)
        await asyncio.sleep(0.02)
    await q.put(None)   # señal de fin

async def consumidor(q, nombre):
    procesados = 0
    while True:
        item = await q.get()
        if item is None:
            await q.task_done()
            break
        procesados += 1
        print(f"{nombre} procesa {item}")
        await q.task_done()
    print(f"{nombre} procesó {procesados} items")

async def main():
    q = asyncio.Queue()
    await asyncio.gather(productor(q), consumidor(q, "C1"), consumidor(q, "C2"))

asyncio.run(main())
print("Concurrencia productor/consumidor completada ✔")

## Ejercicio Independiente: Download Manager concurrente

**Reto:** Construye un **download manager** que descargue (simulado) `N = 20` archivos
limitando la concurrencia a `3` simultáneos. Debes:

1. Crear una corutina `descargar_archivo(id)` que tarde `0.1s` (simulado con `asyncio.sleep`).
2. Presentar un barra/contador de progreso (`x/20 descargados`).
3. **Usar** `asyncio.Semaphore` para que nunca haya más de 3 descargas a la vez.
4. Verificar que el número total descargado es 20 y que la concurrencia máxima ≤ 3.
5. (Opcional) Cronometrar: con límite de 3 y 0.1s por archivo, el total debe ser ≈ 0.7s, no 2s.

Usa el estilo de las celdas anteriores. ¡Aplica lo aprendido!

```
Ejemplo de salida esperada (idea):
   descargando [  3/20 ]
   descargando [  6/20 ]
   ...
   ✔ 20/20 descargados en 0.70s | máx. concurrencia 3
```


In [ ]:
# ==== SOLUCIÓN DEL RETO: Download Manager concurrente ====
import asyncio
import time

N_ARCHIVOS = 20
MAX_CONCURRENCIA = 3

async def descargar_archivo(id_, sem, estado):
    async with sem:
        estado["activas"] += 1
        estado["max"] = max(estado["max"], estado["activas"])
        await asyncio.sleep(0.1)             # I/O simulado
        estado["activas"] -= 1
        estado["hechas"] += 1
        print(f"\r   descargando [ {estado['hechas']:2d}/{N_ARCHIVOS} ]", end="")
    return id_

async def main_download():
    sem = asyncio.Semaphore(MAX_CONCURRENCIA)
    estado = {"activas": 0, "max": 0, "hechas": 0}
    inicio = time.perf_counter()

    resultados = await asyncio.gather(
        *(descargar_archivo(i, sem, estado) for i in range(N_ARCHIVOS))
    )

    duracion = time.perf_counter() - inicio
    print(f"\r✔ {estado['hechas']}/{N_ARCHIVOS} descargados en {duracion:.2f}s | "
          f"máx. concurrencia {estado['max']}")
    return estado, duracion

estado, duracion = asyncio.run(main_download())
assert estado["hechas"] == 20, "Faltaron archivos por descargar"
assert estado["max"] <= 3, "Se superó el límite de concurrencia"
print("\nReto resuelto correctamente ✔")

## Resumen

- **`asyncio`** permite **concurrencia** (no paralelismo) con **un solo hilo**, ideal para I/O.
- **Corutinas** (`async def`) no corren al llamarlas; se ejecutan con `await`, `asyncio.run()` o `create_task()`.
- `asyncio.gather` devuelve resultados **en orden de declaración**; `asyncio.wait` permite
  controlar *cuándo* esperar (`FIRST_COMPLETED`, `ALL_COMPLETED`).
- **Semáforo** limita concurrencia, **Lock** protege secciones críticas, **Queue** + productor/consumidor
  canaliza trabajo, y **timeouts** evitan esperas infinitas.
- Los **protocolos asíncronos** (`async for`, `async with`, generadores async) modelan flujos de I/O.
- **Anti-patrones** a evitar: `time.sleep` en corutinas, olvidar `await`, reusar corutinas cerradas,
  no capturar excepciones en tasks, y esperar secuencialmente dentro de `gather`.

Con estas herramientas puedes construir **clients HTTP concurrentes, web scrapers, download managers,
pipelines de datos y brokers** eficientes en Python.